# Nimbus on an A100 (Colab)

Runs the full Nimbus benchmark suite with Llama-3-8B-Instruct as the target and Llama-3.2-1B-Instruct as the speculative draft. Requires a Colab runtime with an **A100** GPU and a HuggingFace token whose account has accepted the Llama 3 license.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/HridhayP/nimbus.git
%cd nimbus
!pip install -q torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -e '.[dev]' datasets accelerate sentencepiece protobuf

In [ ]:
from huggingface_hub import login
login()  # paste a token with Llama 3 access

In [ ]:
import os
os.environ.update(NIMBUS_MODEL='meta-llama/Meta-Llama-3-8B-Instruct',
                  NIMBUS_DRAFTS='meta-llama/Llama-3.2-1B-Instruct',
                  NIMBUS_RESULTS='results-a100', PYTHONPATH='.')
!bash bench/run_all.sh

In [ ]:
import glob, json
for p in sorted(glob.glob('results-a100/*.json')):
    d = json.load(open(p))
    keys = [k for k in ('output_tok_per_s', 'ttft_ms_median', 'tpot_ms_median', 'tok_per_s_median', 'acceptance_rate', 'perplexity', 'weight_gib') if k in d]
    print(os.path.basename(p), {k: round(d[k], 3) for k in keys})